# SRQ-FLY D4 — CUB five-fresh-seed train-only confirmation

Run all cells in order on a Colab T4 GPU. D4 uses five fresh seeds and never materializes or evaluates CUB test features. A PASS only authorizes protocol review; it is not a held-out result.

In [ ]:
# === Edit path/source values only. Do not edit protocol values below. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'feature/srq-fly-d4-multiseed'
WORK_DIR = '/content/SOHO-CL'
DRIVE_ROOT = '/content/drive/MyDrive/T-SOHO'
DRIVE_TRAIN_CACHE = f'{DRIVE_ROOT}/cub_train_feature_cache'
TRAIN_CACHE_DIR = '/content/cub_train_feature_cache'
D3_OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_cub_d3_train_only_seed2025'
D3_RESULT_PATH = f'{D3_OUTPUT_DIR}/d3_results.json'
D3_AUDIT_PATH = f'{D3_OUTPUT_DIR}/cub_dataset_audit.json'
LOCAL_D3_RESULT = '/content/locked_d3_results.json'
LOCAL_DATASET_AUDIT = '/content/cub_dataset_audit_d4.json'
# WTA infrastructure stays outside the result bundle and resumes on Drive.
CODE_CACHE_ROOT = f'{DRIVE_ROOT}/srq_fly_cub_d4_wta_seed2026_2030'
OUTPUT_DIR = f'{DRIVE_ROOT}/srq_fly_cub_d4_multiseed_seed2026_2030'
CONFIG_SHA256 = 'a0742a545fa83f54b18bcf2372ea6d6e518d214f48f882f2b94696122c2fe8fd'
D3_ARTIFACT_SHA256 = '4d2104c80e3f5fa125839f7723ac86126fbb0395c53b7307a9de1a349b8f380a'
D3_RESULT_SHA256 = 'f172d508c14fd95e7dcece5cd22c04a8e9f88c35c638fa140b476ebf6d4e6f4b'
D3_AUDIT_SHA256 = 'a991b581ed8d78694f75381b9d862a242fcbfd6fe4ad366818161bf700d55125'
TRAIN_CACHE_SHA256 = 'b58dd541f35450d926e1c82826d6491665ca55920a0d5627e2153b65d2015688'

In [ ]:
# Runtime setup. chdir before replacing the repository.
from google.colab import drive
drive.mount('/content/drive')
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
os.chdir('/content')
repo = Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
clone = subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_GIT_URL, WORK_DIR], text=True, capture_output=True)
print(clone.stdout, clone.stderr, sep='')
assert clone.returncode == 0, 'Clone failed. Confirm the D4 branch was pushed.'
os.chdir(WORK_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-kaggle.txt'], check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
config_path = Path('configs/srq_fly_cub_d4_multiseed_train_only.json')
assert hashlib.sha256(config_path.read_bytes()).hexdigest() == CONFIG_SHA256
print('repo commit:', commit)
print('GPU:', torch.cuda.get_device_name(0))
print('locked config:', CONFIG_SHA256)

In [ ]:
# Restore immutable D3 evidence. If absent from Drive, upload the exact D3 ZIP once.
d3_result_source, d3_audit_source = Path(D3_RESULT_PATH), Path(D3_AUDIT_PATH)
if not (d3_result_source.is_file() and d3_audit_source.is_file()):
    from google.colab import files
    print('D3 evidence not found on Drive. Upload srq_fly_cub_d3_train_only.zip.', flush=True)
    uploaded = files.upload()
    assert len(uploaded) == 1, 'Upload exactly one D3 ZIP.'
    uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
    assert uploaded_name.lower().endswith('.zip')
    assert hashlib.sha256(uploaded_bytes).hexdigest() == D3_ARTIFACT_SHA256, 'D3 ZIP SHA-256 mismatch'
    with zipfile.ZipFile(__import__('io').BytesIO(uploaded_bytes)) as archive:
        result_names = [name for name in archive.namelist() if name.endswith('d3_results.json')]
        audit_names = [name for name in archive.namelist() if name.endswith('cub_dataset_audit.json')]
        assert len(result_names) == len(audit_names) == 1
        Path(LOCAL_D3_RESULT).write_bytes(archive.read(result_names[0]))
        Path(LOCAL_DATASET_AUDIT).write_bytes(archive.read(audit_names[0]))
else:
    shutil.copy2(d3_result_source, LOCAL_D3_RESULT)
    shutil.copy2(d3_audit_source, LOCAL_DATASET_AUDIT)
assert hashlib.sha256(Path(LOCAL_D3_RESULT).read_bytes()).hexdigest() == D3_RESULT_SHA256
assert hashlib.sha256(Path(LOCAL_DATASET_AUDIT).read_bytes()).hexdigest() == D3_AUDIT_SHA256
d3 = json.loads(Path(LOCAL_D3_RESULT).read_text())
assert d3['status'] == 'STOP_SRQ_FLY_D3' and d3['uses_test_set'] is False
print('D3 evidence PASS:', D3_RESULT_SHA256)
print('D3 was a formal STOP only; D4 does not relabel it.')

In [ ]:
# Restore the verified TRAIN-only feature cache with visible copy progress.
def copy_file_progress(source, target, chunk=32*2**20):
    source, target = Path(source), Path(target)
    assert source.is_file(), f'Missing required Drive cache file: {source}'
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.is_file() and target.stat().st_size == source.stat().st_size:
        print('RESTORED', target.name, f'{target.stat().st_size/2**20:.1f} MiB')
        return
    partial = target.with_suffix(target.suffix + '.partial')
    partial.unlink(missing_ok=True)
    copied, total, next_report = 0, source.stat().st_size, 10
    with source.open('rb') as reader, partial.open('wb') as writer:
        while block := reader.read(chunk):
            writer.write(block); copied += len(block)
            percent = int(100 * copied / total)
            if percent >= next_report:
                print(f'COPY {target.name}: {percent}% ({copied/2**20:.1f}/{total/2**20:.1f} MiB)', flush=True)
                next_report += 10
    partial.replace(target)
drive_cache, local_cache = Path(DRIVE_TRAIN_CACHE), Path(TRAIN_CACHE_DIR)
copy_file_progress(drive_cache / 'train.pt', local_cache / 'train.pt')
copy_file_progress(drive_cache / 'metadata.json', local_cache / 'metadata.json')
assert not (local_cache / 'test.pt').exists(), 'STOP: CUB test.pt must remain absent'
assert hashlib.sha256((local_cache / 'train.pt').read_bytes()).hexdigest() == TRAIN_CACHE_SHA256
metadata = json.loads((local_cache / 'metadata.json').read_text())
assert metadata['test_features_materialized'] is False and metadata['train_shape'] == [5994, 768]
print('train-only cache PASS:', metadata['train_shape'], '| test.pt absent')

In [ ]:
# D4 correctness gate: synthetic tests only.
test_command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_srq_fly_d4_cub_multiseed.py', 'tests/test_srq_fly_d3_cub.py', 'tests/test_srq_fly_learner.py', 'tests/test_srq_fly_math.py']
subprocess.run(test_command, check=True)
print('SRQ-FLY D4 correctness gate: PASS')

In [ ]:
# Run/resume D4. RAW INNER, CACHE, SEED and OUTER lines expose bounded progress.
output = Path(OUTPUT_DIR); output.mkdir(parents=True, exist_ok=True)
Path(CODE_CACHE_ROOT).mkdir(parents=True, exist_ok=True)
shutil.copy2(LOCAL_D3_RESULT, output / 'locked_d3_results.json')
shutil.copy2(LOCAL_DATASET_AUDIT, output / 'cub_dataset_audit.json')
shutil.copy2(config_path, output / 'locked_config.json')
environment = {'git_commit': commit, 'python': sys.version, 'torch': torch.__version__, 'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0)}
(output / 'environment.json').write_text(json.dumps(environment, indent=2))
command = [sys.executable, '-u', 'tools/srq_fly_d4_cub_multiseed.py', '--config', str(config_path), '--dataset-audit', LOCAL_DATASET_AUDIT, '--feature-cache-dir', TRAIN_CACHE_DIR, '--d3-result', LOCAL_D3_RESULT, '--code-cache-root', CODE_CACHE_ROOT, '--output-dir', OUTPUT_DIR, '--device', 'cuda', '--require-test-hidden']
print('Starting/resuming D4: 40 raw candidates, then 5 paired seed blocks.', flush=True)
print('WTA caches and completed units persist on Drive; rerun this cell after interruption.', flush=True)
started = time.time(); log_path = output / 'd4_run.log'
with log_path.open('a', encoding='utf-8') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True); log.write(line); log.flush()
    returncode = process.wait()
assert returncode == 0, 'D4 runner failed; return the full traceback without editing config.'
assert (output / 'd4_results.json').is_file()
print(f'D4 process COMPLETE in {(time.time()-started)/60:.1f} minutes')

In [ ]:
# Display train-only evidence and download the compact result bundle. STOP afterwards.
import pandas as pd
result = json.loads((Path(OUTPUT_DIR) / 'd4_results.json').read_text())
rows = []
for item in result['seed_results']:
    comparison = item['comparison']
    rows.append({'seed': item['seed'], 'SRQ_AA': item['srq_fly_10000']['validation_average_accuracy'], 'FLY4518_AA': item['exact_fly_4518']['validation_average_accuracy'], 'raw_AA': item['raw_ridge']['validation_average_accuracy'], 'SRQ_minus_FLY4518_pp': comparison['srq_average_gain_over_state_matched_fly_pp'], 'SRQ_state_bytes': item['srq_fly_10000']['persistent_state_bytes'], 'agreement': comparison['minimum_prediction_agreement']})
display(pd.DataFrame(rows))
print('status:', result['status'])
print('selected raw Ridge lambda:', result['selected_raw_ridge_lambda'])
print('summaries:', json.dumps(result['summaries'], indent=2))
print('gates:', json.dumps(result['gates'], indent=2))
archive = shutil.make_archive('/content/srq_fly_cub_d4_multiseed_train_only', 'zip', root_dir=OUTPUT_DIR)
print('artifact SHA-256:', hashlib.sha256(Path(archive).read_bytes()).hexdigest())
from google.colab import files
files.download(archive)
print('STOP. Return the ZIP for audit. Do not materialize or evaluate CUB test features.')